# Bee Orientation Estimation

Trains and evaluates CNNs that predict the orientation $\theta$ of a bee
centered in an image patch (produced by `preprocessing.py`).

**Key idea:** instead of regressing $\theta$ directly (which is discontinuous
at the $\pm\pi$ wrap-around), the model outputs two continuous values
$(\sin\theta, \cos\theta)$. The orientation is reconstructed via
$\hat\theta = \operatorname{atan2}(\widehat{\sin\theta}, \widehat{\cos\theta})$.

- **Loss:** MSE over the $(\sin\theta, \cos\theta)$ output pair.
- **Error metric:** signed angular error
  $\Delta\theta = \operatorname{atan2}(\sin(\theta - \hat\theta), \cos(\theta - \hat\theta))$,
  reported as mean/median absolute error in degrees.
- **Models:** ResNet-18, ResNet-50, MobileNetV4 (all via `timm`).

In [ ]:
# Uncomment on first run if dependencies are missing on the HPC node:
# %pip install --user timm torch torchvision pandas matplotlib

In [ ]:
from __future__ import annotations

import math
import random
from dataclasses import dataclass, field
from pathlib import Path
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

## Configuration

Everything tunable lives in a single `Config` dataclass.

In [ ]:
@dataclass
class Config:
    # --- data ---
    data_dir: Path = Path("/scratch/cvcdt011/data")
    crops_subdir: str = "crops"
    trajectories_subdir: str = "rec1_trajectories"
    angles_in_degrees: bool = False   # set True if column 4 of the trajectory files is in degrees
    expected_crop_size: int = 100     # 2 * half_size from preprocessing.py
    filter_partial_crops: bool = True # drop border crops smaller than expected_crop_size

    # --- split ---
    split_strategy: str = "trajectory"  # "trajectory" (no temporal leakage) or "random"
    val_fraction: float = 0.15
    test_fraction: float = 0.15
    seed: int = 42

    # --- input pipeline ---
    image_size: int = 224
    batch_size: int = 64
    num_workers: int = 4
    # Rotation augmentation rotates the patch and adjusts theta accordingly.
    # It assumes theta is measured as atan2(dy, dx) in image coordinates (y pointing
    # down). Verify the label convention before enabling, otherwise labels get corrupted.
    rotation_augment: bool = False

    # --- models / training ---
    model_names: tuple[str, ...] = ("resnet18", "resnet50", "mobilenetv4_conv_medium")
    pretrained: bool = True
    epochs: int = 15
    lr: float = 3e-4
    weight_decay: float = 1e-4
    use_amp: bool = True
    checkpoint_dir: Path = Path("checkpoints")

    @property
    def crops_dir(self) -> Path:
        return self.data_dir / self.crops_subdir

    @property
    def trajectories_dir(self) -> Path:
        return self.data_dir / self.trajectories_subdir


cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(cfg.seed)

## Sample index

Each trajectory file `rec1_trajectories/<traj_id>.txt` holds rows
`frame_nb, position_x, position_y, object_class, orientation_angle`.
A sample pairs the crop
`crops/<traj_id>/frame_<frame>.png` with its orientation label.

In [ ]:
class Sample(NamedTuple):
    path: Path
    theta: float  # radians
    traj_id: str


def load_samples(cfg: Config) -> list[Sample]:
    """Build the (crop path, orientation) index from the trajectory files."""
    samples: list[Sample] = []
    for traj_file in sorted(cfg.trajectories_dir.glob("*.txt")):
        traj_id = traj_file.stem
        crop_dir = cfg.crops_dir / traj_id
        if not crop_dir.exists():
            continue
        for line in traj_file.read_text().splitlines():
            parts = line.strip().split(",")
            if len(parts) < 5:
                continue
            frame = int(parts[0])
            theta = float(parts[4])
            if cfg.angles_in_degrees:
                theta = math.radians(theta)
            path = crop_dir / f"frame_{frame:06d}.png"
            if path.exists():
                samples.append(Sample(path=path, theta=theta, traj_id=traj_id))
    return samples


def drop_partial_crops(samples: list[Sample], expected_size: int) -> list[Sample]:
    """Remove crops clipped at the frame border (smaller than expected_size)."""
    kept: list[Sample] = []
    for s in samples:
        with Image.open(s.path) as img:
            if img.size == (expected_size, expected_size):
                kept.append(s)
    return kept


samples = load_samples(cfg)
print(f"Indexed {len(samples)} samples from {len({s.traj_id for s in samples})} trajectories")
if cfg.filter_partial_crops:
    samples = drop_partial_crops(samples, cfg.expected_crop_size)
    print(f"{len(samples)} samples after dropping partial border crops")

## Train / val / test split

Default splits **by trajectory**: consecutive frames of the same bee are nearly
identical, so a random frame-level split would leak information from train to test.

In [ ]:
def split_samples(
    samples: list[Sample], cfg: Config
) -> tuple[list[Sample], list[Sample], list[Sample]]:
    rng = random.Random(cfg.seed)
    if cfg.split_strategy == "trajectory":
        traj_ids = sorted({s.traj_id for s in samples})
        rng.shuffle(traj_ids)
        n = len(traj_ids)
        n_test = max(1, round(n * cfg.test_fraction))
        n_val = max(1, round(n * cfg.val_fraction))
        test_ids = set(traj_ids[:n_test])
        val_ids = set(traj_ids[n_test : n_test + n_val])
        train = [s for s in samples if s.traj_id not in test_ids | val_ids]
        val = [s for s in samples if s.traj_id in val_ids]
        test = [s for s in samples if s.traj_id in test_ids]
    elif cfg.split_strategy == "random":
        shuffled = samples.copy()
        rng.shuffle(shuffled)
        n = len(shuffled)
        n_test = round(n * cfg.test_fraction)
        n_val = round(n * cfg.val_fraction)
        test = shuffled[:n_test]
        val = shuffled[n_test : n_test + n_val]
        train = shuffled[n_test + n_val :]
    else:
        raise ValueError(f"Unknown split strategy: {cfg.split_strategy!r}")
    return train, val, test


train_samples, val_samples, test_samples = split_samples(samples, cfg)
print(f"train: {len(train_samples)}  val: {len(val_samples)}  test: {len(test_samples)}")

## Dataset & dataloaders

Targets are `[sin(theta), cos(theta)]` (this order is assumed everywhere,
including `atan2(pred[:, 0], pred[:, 1])` at decode time).

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class BeeOrientationDataset(Dataset):
    def __init__(self, samples: list[Sample], cfg: Config, augment: bool = False) -> None:
        self.samples = samples
        self.augment = augment
        self.rotation_augment = augment and cfg.rotation_augment
        self.transform = transforms.Compose(
            [
                transforms.Resize((cfg.image_size, cfg.image_size)),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ]
        )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        sample = self.samples[idx]
        img = Image.open(sample.path).convert("RGB")
        theta = sample.theta

        if self.rotation_augment:
            angle_deg = random.uniform(0.0, 360.0)
            # PIL rotates counter-clockwise on screen; with image coordinates
            # (y down) this decreases theta = atan2(dy, dx) by the same amount.
            img = img.rotate(angle_deg, resample=Image.BILINEAR)
            theta = theta - math.radians(angle_deg)

        target = torch.tensor([math.sin(theta), math.cos(theta)], dtype=torch.float32)
        return self.transform(img), target


def make_loaders(
    train_samples: list[Sample],
    val_samples: list[Sample],
    test_samples: list[Sample],
    cfg: Config,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    def loader(ds: Dataset, shuffle: bool) -> DataLoader:
        return DataLoader(
            ds,
            batch_size=cfg.batch_size,
            shuffle=shuffle,
            num_workers=cfg.num_workers,
            pin_memory=True,
            drop_last=False,
        )

    return (
        loader(BeeOrientationDataset(train_samples, cfg, augment=True), shuffle=True),
        loader(BeeOrientationDataset(val_samples, cfg), shuffle=False),
        loader(BeeOrientationDataset(test_samples, cfg), shuffle=False),
    )


train_loader, val_loader, test_loader = make_loaders(
    train_samples, val_samples, test_samples, cfg
)

## Sanity check: patches with their orientation labels

The arrow is drawn assuming theta = atan2(dy, dx) in image coordinates
(y down). If the arrows do not align with the bees' body axes, the label
convention differs — adjust before training.

In [ ]:
def show_labeled_patches(samples: list[Sample], n: int = 8) -> None:
    picks = random.sample(samples, min(n, len(samples)))
    fig, axes = plt.subplots(1, len(picks), figsize=(2.2 * len(picks), 2.6))
    for ax, s in zip(np.atleast_1d(axes), picks):
        img = Image.open(s.path).convert("RGB")
        w, h = img.size
        cx, cy = w / 2, h / 2
        r = 0.4 * min(w, h)
        ax.imshow(img)
        ax.arrow(
            cx, cy, r * math.cos(s.theta), r * math.sin(s.theta),
            color="red", width=1.0, head_width=5.0, length_includes_head=True,
        )
        ax.set_title(f"{math.degrees(s.theta):.0f}°", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_labeled_patches(train_samples)

## Models

All three architectures come from `timm` with the classification head replaced
by a 2-unit linear layer predicting `[sin(theta), cos(theta)]`.

In [ ]:
def create_model(name: str, cfg: Config) -> nn.Module:
    return timm.create_model(name, pretrained=cfg.pretrained, num_classes=2)

## Loss & metrics

In [ ]:
def angles_from_sincos(output: torch.Tensor) -> torch.Tensor:
    """Decode theta from a (N, 2) [sin, cos] prediction."""
    return torch.atan2(output[:, 0], output[:, 1])


def signed_angular_error(theta_true: torch.Tensor, theta_pred: torch.Tensor) -> torch.Tensor:
    """Wrap-around-safe angular error in (-pi, pi]."""
    diff = theta_true - theta_pred
    return torch.atan2(torch.sin(diff), torch.cos(diff))


def angular_metrics(theta_true: np.ndarray, theta_pred: np.ndarray) -> dict[str, float]:
    err = signed_angular_error(torch.from_numpy(theta_true), torch.from_numpy(theta_pred))
    abs_err_deg = torch.rad2deg(err.abs())
    return {
        "mae_deg": abs_err_deg.mean().item(),
        "median_ae_deg": abs_err_deg.median().item(),
        "rmse_deg": torch.sqrt(torch.rad2deg(err).pow(2).mean()).item(),
    }

## Training & evaluation loops

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: torch.cuda.amp.GradScaler,
    cfg: Config,
) -> float:
    model.train()
    total_loss, n_samples = 0.0, 0
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=cfg.use_amp):
            outputs = model(images)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
        n_samples += images.size(0)
    return total_loss / n_samples


@torch.no_grad()
def evaluate(
    model: nn.Module, loader: DataLoader, criterion: nn.Module, cfg: Config
) -> tuple[dict[str, float], np.ndarray, np.ndarray]:
    """Returns (metrics, theta_true, theta_pred) over the whole loader."""
    model.eval()
    total_loss, n_samples = 0.0, 0
    trues: list[torch.Tensor] = []
    preds: list[torch.Tensor] = []
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=cfg.use_amp):
            outputs = model(images)
            loss = criterion(outputs, targets)
        total_loss += loss.item() * images.size(0)
        n_samples += images.size(0)
        trues.append(angles_from_sincos(targets.float()).cpu())
        preds.append(angles_from_sincos(outputs.float()).cpu())
    theta_true = torch.cat(trues).numpy()
    theta_pred = torch.cat(preds).numpy()
    metrics = {"mse": total_loss / n_samples, **angular_metrics(theta_true, theta_pred)}
    return metrics, theta_true, theta_pred


@dataclass
class RunResult:
    model_name: str
    history: pd.DataFrame
    test_metrics: dict[str, float]
    theta_true: np.ndarray
    theta_pred: np.ndarray
    checkpoint_path: Path


def train_and_evaluate(
    name: str,
    cfg: Config,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
) -> RunResult:
    print(f"\n=== {name} ===")
    seed_everything(cfg.seed)
    model = create_model(name, cfg).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp and device.type == "cuda")

    cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = cfg.checkpoint_dir / f"{name}_best.pt"

    history: list[dict[str, float]] = []
    best_val_mae = float("inf")

    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, cfg)
        val_metrics, _, _ = evaluate(model, val_loader, criterion, cfg)
        scheduler.step()

        history.append({"epoch": epoch, "train_mse": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}})
        marker = ""
        if val_metrics["mae_deg"] < best_val_mae:
            best_val_mae = val_metrics["mae_deg"]
            torch.save(model.state_dict(), checkpoint_path)
            marker = "  *"
        print(
            f"epoch {epoch:3d} | train MSE {train_loss:.4f} | "
            f"val MSE {val_metrics['mse']:.4f} | val MAE {val_metrics['mae_deg']:6.2f}°{marker}"
        )

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    test_metrics, theta_true, theta_pred = evaluate(model, test_loader, criterion, cfg)
    print(f"test: MSE {test_metrics['mse']:.4f} | MAE {test_metrics['mae_deg']:.2f}° | median {test_metrics['median_ae_deg']:.2f}°")

    return RunResult(
        model_name=name,
        history=pd.DataFrame(history),
        test_metrics=test_metrics,
        theta_true=theta_true,
        theta_pred=theta_pred,
        checkpoint_path=checkpoint_path,
    )

## Run all experiments

In [ ]:
results: dict[str, RunResult] = {}
for name in cfg.model_names:
    results[name] = train_and_evaluate(name, cfg, train_loader, val_loader, test_loader)

## Model comparison

In [ ]:
comparison = pd.DataFrame(
    {name: res.test_metrics for name, res in results.items()}
).T.rename_axis("model")
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, res in results.items():
    axes[0].plot(res.history["epoch"], res.history["train_mse"], label=f"{name} (train)")
    axes[0].plot(res.history["epoch"], res.history["val_mse"], "--", label=f"{name} (val)")
    axes[1].plot(res.history["epoch"], res.history["val_mae_deg"], label=name)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("MSE"); axes[0].set_title("sin/cos MSE loss"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val MAE [°]"); axes[1].set_title("Validation angular error"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(4.5 * len(results), 3.5), squeeze=False)
for ax, (name, res) in zip(axes[0], results.items()):
    err_deg = np.degrees(
        signed_angular_error(torch.from_numpy(res.theta_true), torch.from_numpy(res.theta_pred)).numpy()
    )
    ax.hist(err_deg, bins=72, range=(-180, 180))
    ax.set_title(f"{name}\nMAE {res.test_metrics['mae_deg']:.1f}°")
    ax.set_xlabel("signed angular error [°]")
plt.tight_layout()
plt.show()

## Qualitative results

Red arrow: ground truth. Blue arrow: prediction of the best model (lowest test MAE).

In [ ]:
best_name = comparison["mae_deg"].idxmin()
best_res = results[best_name]
print(f"Best model: {best_name}")

best_model = create_model(best_name, cfg).to(device)
best_model.load_state_dict(torch.load(best_res.checkpoint_path, map_location=device))
best_model.eval()


@torch.no_grad()
def show_predictions(samples: list[Sample], model: nn.Module, cfg: Config, n: int = 8) -> None:
    picks = random.sample(samples, min(n, len(samples)))
    ds = BeeOrientationDataset(picks, cfg)
    images = torch.stack([ds[i][0] for i in range(len(picks))]).to(device)
    theta_pred = angles_from_sincos(model(images).float()).cpu().numpy()

    fig, axes = plt.subplots(1, len(picks), figsize=(2.2 * len(picks), 2.6))
    for ax, s, tp in zip(np.atleast_1d(axes), picks, theta_pred):
        img = Image.open(s.path).convert("RGB")
        w, h = img.size
        cx, cy = w / 2, h / 2
        r = 0.4 * min(w, h)
        ax.imshow(img)
        ax.arrow(cx, cy, r * math.cos(s.theta), r * math.sin(s.theta),
                 color="red", width=1.0, head_width=5.0, length_includes_head=True)
        ax.arrow(cx, cy, r * math.cos(tp), r * math.sin(tp),
                 color="deepskyblue", width=1.0, head_width=5.0, length_includes_head=True)
        err = math.degrees(math.atan2(math.sin(s.theta - tp), math.cos(s.theta - tp)))
        ax.set_title(f"err {err:+.1f}°", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_predictions(test_samples, best_model, cfg)